
## Overview
This notebook shows how to create and query a table or DataFrame from a file uploaded to DBFS, a Databricks File System for storing data. It is written in Python, so the default cell type is Python.

In [0]:
# File location and type
file_location = "/FileStore/tables/telco__1_.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

display(df.sample(withReplacement=False, fraction=500/df.count()))

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
9763-GRSKD,Male,0,Yes,Yes,13,Yes,No,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Mailed check,49.95,587.45,No
9959-WOFKT,Male,0,No,Yes,71,Yes,Yes,Fiber optic,Yes,No,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),106.7,7382.25,No
8665-UTDHZ,Male,0,Yes,Yes,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,30.2,30.2,Yes
9489-DEDVP,Female,0,Yes,Yes,70,Yes,Yes,DSL,Yes,Yes,No,No,Yes,No,Two year,Yes,Credit card (automatic),69.2,4872.35,No
5698-BQJOH,Female,0,No,No,9,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,94.4,857.25,Yes
3887-PBQAO,Female,0,Yes,Yes,45,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),25.9,1216.6,No
8402-OOOHJ,Female,0,No,No,41,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.65,835.15,No
7799-LGRDP,Female,0,No,No,43,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card (automatic),25.7,1188.2,No
5716-EZXZN,Female,0,Yes,Yes,65,Yes,Yes,Fiber optic,Yes,Yes,No,Yes,Yes,No,Two year,Yes,Credit card (automatic),99.05,6416.7,No
6837-BJYDQ,Male,0,No,No,3,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,19.6,61.35,No


In [0]:
# Create a view or table

temp_table_name = "telco__1__csv"

df.createOrReplaceTempView(temp_table_name)

In [0]:
%sql

/* Query the created temp table in a SQL cell */

select * from `telco__1__csv`

In [0]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.

permanent_table_name = "telco__1__csv"

# df.write.format("parquet").saveAsTable(permanent_table_name)

In [0]:
customenerID_MonthlyCharges = df.select("customerID", "MonthlyCharges")
display(customenerID_MonthlyCharges.sample(withReplacement=False, fraction=500/customenerID_MonthlyCharges.count()))

customerID,MonthlyCharges
7892-POOKP,104.8
4190-MFLUW,55.2
5248-YGIJN,90.25
8168-UQWWF,97.85
5122-CYFXA,75.3
4445-ZJNMU,99.3
7233-PAHHL,84.0
9848-JQJTX,100.9
7123-WQUHX,95.0
0486-HECZI,96.75


In [0]:
contract_proportions = df.groupBy("Contract").count()
display(contract_proportions)

Contract,count
Month-to-month,3875
One year,1473
Two year,1695


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F  
MultipleLine_df = df.select("MultipleLines")
customers_MultipleLines = df.filter(F.col("MultipleLines") == "Yes").count()  # Используем F.col()
result_value = customers_MultipleLines
display(f"Amount of customers with Multiple Lines: {result_value}")

'Amount of customers with Multiple Lines: 2971'

In [0]:
values_tenure = df.selectExpr("min(tenure) as min_tenure", "max(tenure) as max_tenure").first()
display(f"Minimum Tenure: {values_tenure['min_tenure']}")
display(f"Maximum Tenure: {values_tenure['max_tenure']}")


'Minimum Tenure: 0''Maximum Tenure: 72'

In [0]:
from pyspark.sql.functions import avg, col
monthly_charges_gender = df.groupBy(col("gender")).agg(avg(col("MonthlyCharges")))
display(monthly_charges_gender)


gender,avg(MonthlyCharges)
Female,65.20424311926602
Male,64.32748241912773


In [0]:
year_contracts = df.filter(df["Contract"] == "One year").agg(avg("MonthlyCharges")).collect()[0][0]
years_month_contracts = df.filter((df["Contract"] == "Two year") | (df["Contract"] == "Month-to-month")).agg(avg("MonthlyCharges")).collect()[0][0]
comparison = year_contracts > years_month_contracts
display("1 year customers pay more ", comparison)

'1 year customers pay more 'True

In [0]:
from pyspark.sql.functions import col
df = df.withColumn("AverageCharges", col("TotalCharges") / col("tenure"))
df_AvgCh = df.select("AverageCharges")
count_rows = df_AvgCh.count()
fraction = min(1.0, 500 / count_rows) 
df_sampled = df_AvgCh.sample(withReplacement=False, fraction=fraction)
display(df_sampled)


AverageCharges
54.075
75.825
52.835
47.57
90.845
66.8830985915493
100.58
19.96484375
70.45625
24.735211267605635
